In [174]:
from pathlib import Path
import requests
from bs4 import BeautifulSoup
import re
import numpy as np
import pandas as pd

def row_label(th):
    a = th.find("a")
    return a.get_text(strip=True) if a else th.get_text(separator=" ", strip=True).split(",")[0].strip()


def parse_financial_table(html, names):
    """names — подписи из первого столбца (текст ссылки), например ['Выручка', 'EBITDA']."""
    want = set(names)
    soup = BeautifulSoup(html, "html.parser")
    table = soup.find("table", class_="financials")
    years = [s.get_text(strip=True) for s in table.find("tr", class_="header_row").find_all("strong")]
    out = {}
    for tr in table.find_all("tr", attrs={"field": True}):
        lab = row_label(tr.find("th"))
        if lab not in want:
            continue
        tds = tr.find_all("td")
        vals = [t if (t := td.get_text(strip=True)) else None for td in tds[1:]]
        out[lab] = dict(zip(years, vals))
    return out

In [175]:
stock_list = requests.get("https://smart-lab.ru/q/shares_fundamental5/")

stock_report_pattern = r'/q/\w+/f/y/'
stock_report_pages = [
    (stock.split("/")[2], f"https://smart-lab.ru/{stock}MSFO/")
        for stock in re.findall(stock_report_pattern, stock_list.text)
]
stock_report_pages = [
    (stock, requests.get(stock_url))
        for stock, stock_url in stock_report_pages
]

In [176]:
fields = [
    "Выручка",
    "Операционная прибыль",
    "EBITDA",
    "Чистая прибыль",
    "Опер.денежный поток",
    "CAPEX",
    "FCF",
    "Див доход, ао",
    "Дивиденды/прибыль",
    "Опер. расходы",
    "Себестоимость",
    "Амортизация",
    "Расх на персонал",
    "Процентные расходы",
    "Активы",
    "Чистые активы",
    "Долг",
    "Наличность	",
    "Чистый долг",
    "Цена акции ао",
    "Число акций ао",
    "Капитализация",
    "EV",
    "Баланс стоимость",
    "EPS",
    "FCF/акцию",
    "BV/акцию",
    "Рентаб EBITDA",
    "Чистая рентаб",
    "Доходность FCF",
    "ROE",
    "ROA",
    "P/E",
    "P/FCF",
    "P/S",
    "P/BV",
    "EV/EBITDA",
    "Долг/EBITDA",
    "R&D/CAPEX",
    "CAPEX/Выручка"
]
available_date_field = ["2020", "2021", "2022", "2023", "2024", "2025", "2026"]

In [195]:
dataset = []
for stock, html in stock_report_pages:
    df = {}
    parsed = parse_financial_table(html.text, fields)
    dates = [int(d) for d in list(parsed.values())[0] if d in available_date_field]
    df['ticker'] = [stock] * len(dates)
    df['year'] = dates
    for field in parsed:
        df[field] = [
            float(parsed[field][d].replace("%", "").replace(" ", "")) if parsed[field][d] is not None else np.nan
                for d in parsed[field] if d in available_date_field
        ]
    dataset.append(pd.DataFrame(df))
ds = pd.concat(dataset)

In [196]:
ds["Див доход, ао"].fillna(0.0)
ds["total_price"] = (1 + ds["Див доход, ао"] / 100) * ds["Цена акции ао"]
ds["total_price"].isna().sum()
ds = ds[~ds["total_price"].isna()]

In [197]:
target_ds = ds[["ticker", "year", "total_price"]].copy()
target_ds["year"] = target_ds["year"] - 1
momentum_ds = ds[["ticker", "year", "total_price"]].copy()
momentum_ds["year"] = momentum_ds["year"] + 1

In [198]:
ds = pd.merge(ds, target_ds, on=["ticker", "year"], suffixes=(None, '_target'))
ds["revenue"] = np.log(ds["total_price_target"] / ds["total_price"])
ds = pd.merge(ds, momentum_ds, on=["ticker", "year"], suffixes=(None, '_momentum'))
ds["momentum"] = np.log(ds["total_price"] / ds["total_price_momentum"])

In [199]:
ds = ds[
    ~ds["revenue"].isna() 
    & ~ds["momentum"].isna()
    & ~ds["P/E"].isna()
    & ~ds["ROE"].isna()
    & ~ds["P/BV"].isna()
    & ~ds["EV/EBITDA"].isna()
    & ~ds["Долг/EBITDA"].isna()
    & ~ds["R&D/CAPEX"].isna()
    & ~ds["CAPEX/Выручка"].isna()
]

In [200]:
ds[[
    "momentum",
    "P/E",
    "ROE",
    "P/BV",
    "EV/EBITDA",
    "Долг/EBITDA",
    "R&D/CAPEX",
    "CAPEX/Выручка",
]].corr()

,momentum,P/E,ROE,P/BV,EV/EBITDA,Долг/EBITDA,R&D/CAPEX,CAPEX/Выручка
momentum,1.000000,0.015354,0.030832,0.028912,0.025850,-0.006954,-0.019056,-0.030973
P/E,0.015354,1.000000,-0.005017,0.586001,0.790750,0.011236,-0.005425,-0.011164
ROE,0.030832,-0.005017,1.000000,0.126149,-0.021509,-0.009693,-0.031812,0.008852
P/BV,0.028912,0.586001,0.126149,1.000000,0.516212,-0.020612,-0.013716,-0.005614
EV/EBITDA,0.025850,0.790750,-0.021509,0.516212,1.000000,0.265437,-0.015294,-0.001884
Долг/EBITDA,-0.006954,0.011236,-0.009693,-0.020612,0.265437,1.000000,-0.023877,0.024024
R&D/CAPEX,-0.019056,-0.005425,-0.031812,-0.013716,-0.015294,-0.023877,1.000000,-0.011170
CAPEX/Выручка,-0.030973,-0.011164,0.008852,-0.005614,-0.001884,0.024024,-0.011170,1.000000


In [201]:
ds.to_csv("dataset.csv")